In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

from sklearn.metrics import r2_score, mean_absolute_error

In [10]:
df = pd.read_csv("zameen-updated.csv/zameen-updated.csv")

In [11]:
df.head()

,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent,Area Type,Area Size,Area Category
0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,4 Marla,For Sale,2,02-04-2019,NaN,NaN,Marla,4.0,0-5 Marla
1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,5.6 Marla,For Sale,3,05-04-2019,NaN,NaN,Marla,5.6,5-10 Marla
2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,8 Marla,For Sale,5,07-17-2019,NaN,NaN,Marla,8.0,5-10 Marla
3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,2 Kanal,For Sale,4,04-05-2019,NaN,NaN,Kanal,2.0,1-5 Kanal
4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,For Sale,3,07-10-2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director,Marla,8.0,5-10 Marla


In [12]:
# 3. DATA CLEANING

In [13]:
# Drop useless columns
df.drop(['property_id', 'page_url', 'location_id', 'area'], axis=1, inplace=True)

In [17]:
df.isnull().sum()

property_type    0
price            0
location         0
city             0
province_name    0
latitude         0
longitude        0
baths            0
purpose          0
bedrooms         0
date_added       0
agency           0
agent            0
Area Type        0
Area Size        0
Area Category    0
dtype: int64

In [14]:
# Convert date
df['date_added'] = pd.to_datetime(df['date_added'])


In [15]:
# Fill missing values
df['agency'].fillna("Unknown", inplace=True)
df['agent'].fillna("Unknown", inplace=True)

In [18]:
# FEATURE ENGINEERING

In [19]:
### 4.1 Property Features

In [20]:
df['total_rooms'] = df['bedrooms'] + df['baths']
df['room_density'] = df['total_rooms'] / (df['Area Size'] + 1)
df['area_per_room'] = df['Area Size'] / (df['total_rooms'] + 1)
df['luxury_score'] = df['bedrooms'] * df['baths']

In [21]:
### 4.2 Time Features

In [22]:
df['year'] = df['date_added'].dt.year
df['month'] = df['date_added'].dt.month

In [23]:
### 4.3 Location Intelligence

In [24]:
city_counts = df['city'].value_counts().to_dict()
df['city_frequency'] = df['city'].map(city_counts)

In [25]:
# 4.4 Encode Categorical Features

In [26]:
le = LabelEncoder()

categorical_cols = ['property_type', 'location', 'city', 'province_name',
                    'purpose', 'agency', 'agent', 'Area Type', 'Area Category']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [27]:
# 5 DEFINE FEATURES & TARGET

In [28]:
X = df.drop(['price', 'date_added'], axis=1)
y = df['price']

In [29]:
# 6. TRAIN MODEL (BEFORE FEATURE SELECTION)

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_before = RandomForestRegressor(n_estimators=100, random_state=42)
model_before.fit(X_train, y_train)

y_pred_before = model_before.predict(X_test)

print("=== BEFORE FEATURE SELECTION ===")
print("R2 Score:", r2_score(y_test, y_pred_before))
print("MAE:", mean_absolute_error(y_test, y_pred_before))

=== BEFORE FEATURE SELECTION ===
R2 Score: 0.8698067895164475
MAE: 2927622.489315883


In [31]:
# 7. FEATURE SELECTION

In [32]:
### 7.1 Feature Importance

In [33]:
importances = pd.Series(model_before.feature_importances_, index=X.columns)
top_features = importances.nlargest(15).index

X = X[top_features]

In [34]:
### 7.2 Correlation Removal

In [35]:
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
X.drop(columns=to_drop, inplace=True)

In [36]:
### 7.3 RFE

In [37]:
model = LinearRegression()

rfe = RFE(model, n_features_to_select=10)
rfe.fit(X, y)

selected_features = X.columns[rfe.support_]
X = X[selected_features]

In [38]:
# FINAL MODEL (AFTER FEATURE ENGINEERING)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_after = RandomForestRegressor(n_estimators=200, random_state=42)
model_after.fit(X_train, y_train)

y_pred_after = model_after.predict(X_test)

print("\n=== AFTER FEATURE ENGINEERING ===")
print("R2 Score:", r2_score(y_test, y_pred_after))
print("MAE:", mean_absolute_error(y_test, y_pred_after))

In [ ]:
### 9. FINAL COMPARISON

In [ ]:
print("\n===== FINAL COMPARISON =====")
print("Before R2:", r2_score(y_test, y_pred_before))
print("After R2:", r2_score(y_test, y_pred_after))

In [ ]:
# 10. VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt

scores = [r2_score(y_test, y_pred_before), r2_score(y_test, y_pred_after)]
labels = ["Before", "After"]

plt.bar(labels, scores)
plt.title("Model Improvement After Feature Engineering")
plt.ylabel("R2 Score")
plt.show()